# Train Contra PPO trên Kaggle

Chạy lần lượt từng cell từ trên xuống. Notebook này nhúng trực tiếp các phần cần thiết từ `train.py`, `Contra/actions.py`, `Contra/contra_env.py`, và `Contra/wrappers.py`, nên trên Kaggle bạn chỉ cần gắn ROM `contra.nes` dưới dạng Kaggle Dataset/Input.

Nếu resume từ checkpoint, gắn thêm file `.zip` checkpoint vào Kaggle Input rồi sửa `RESUME_PATH` trong cell cấu hình.


In [ ]:
# Cell 1 - Cài/check dependencies
# Nếu Kaggle notebook bật Internet, cell này sẽ tự cài package còn thiếu.
# Nếu Internet tắt, hãy thêm các wheel/package tương ứng vào Kaggle Input rồi cài thủ công.

import importlib.util
import subprocess
import sys

packages = {
    "stable_baselines3": "stable-baselines3[extra]",
    "gymnasium": "gymnasium",
    "nes_py": "nes-py",
    "cv2": "opencv-python-headless",
}

missing = [pip_name for module_name, pip_name in packages.items() if importlib.util.find_spec(module_name) is None]
if missing:
    print("Installing missing packages:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("All required packages are available.")


In [ ]:
# Cell 2 - Cấu hình Kaggle paths và hyperparameters

import glob
import os
import random
import warnings
from pathlib import Path

import numpy as np

try:
    import torch
except Exception:
    torch = None

warnings.filterwarnings("ignore")

KAGGLE_WORKING = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
LOG_DIR = str(KAGGLE_WORKING / "logs")
CHECKPOINT_DIR = str(KAGGLE_WORKING / "checkpoints")
BEST_MODEL_DIR = str(KAGGLE_WORKING / "best_model")

for path in [LOG_DIR, CHECKPOINT_DIR, BEST_MODEL_DIR]:
    os.makedirs(path, exist_ok=True)

# Kaggle: upload/gắn ROM vào Input, notebook sẽ tự tìm /kaggle/input/**/contra.nes.
# Có thể override bằng os.environ["CONTRA_ROM_PATH"] hoặc sửa trực tiếp ROM_PATH bên dưới.
def find_rom_path():
    candidates = []
    env_path = os.environ.get("CONTRA_ROM_PATH")
    if env_path:
        candidates.append(env_path)
    candidates.extend(glob.glob("/kaggle/input/**/contra.nes", recursive=True))
    candidates.extend(glob.glob("./**/contra.nes", recursive=True))
    for candidate in candidates:
        if os.path.exists(candidate):
            return os.path.abspath(candidate)
    raise FileNotFoundError(
        "Không tìm thấy contra.nes. Hãy gắn ROM vào Kaggle Input hoặc set CONTRA_ROM_PATH."
    )

ROM_PATH = find_rom_path()
print("ROM_PATH:", ROM_PATH)
print("LOG_DIR:", LOG_DIR)
print("CHECKPOINT_DIR:", CHECKPOINT_DIR)
print("BEST_MODEL_DIR:", BEST_MODEL_DIR)

# Training config tương đương train.py, nhưng mặc định nhẹ hơn cho Kaggle notebook.
N_ENVS = 4
TOTAL_TIMESTEPS = 5_000_000
SEED = 42
MAX_EPISODE_STEPS = 4500
FRAME_SKIP = 4

# Sửa thành đường dẫn checkpoint .zip nếu muốn resume, ví dụ:
# RESUME_PATH = "/kaggle/input/my-checkpoints/contra_ppo_500000_steps.zip"
RESUME_PATH = None

# Dùng progress bar tự quản bằng tqdm thay vì progress_bar=True của SB3.
# SB3 dùng rich live display, dễ bị kẹt sau khi interrupt trên Kaggle.
USE_PROGRESS_BAR = True

# Chạy test nhanh trước train để bắt lỗi ROM/env sớm.
SMOKE_TEST_STEPS = 8

random.seed(SEED)
np.random.seed(SEED)
if torch is not None:
    torch.manual_seed(SEED)


In [ ]:
# Cell 3 - Action space giống Contra/actions.py

COMPLEX_MOVEMENT = [
    ['NOOP'],
    ['right'],
    ['right', 'A'],
    ['right', 'B'],
    ['right', 'A', 'up'],
    ['right', 'B', 'up'],
    ['right', 'A', 'B', 'up'],
    ['A'],
    ['B'],
    ['A', 'B'],

    ['left'],
    ['left', 'A'],
    ['left', 'B'],
    ['left', 'A', 'up'],
    ['left', 'B', 'up'],
    ['left', 'A', 'B', 'up'],

    ['down', 'A'],
    ['down', 'B'],
    ['down', 'A', 'B'],
    ['up', 'A'],
    ['up', 'A', 'B'],
]


In [ ]:
# Cell 4 - ContraEnv nhúng trực tiếp từ Contra/contra_env.py

import numpy as np
from nes_py import NESEnv
import nes_py._rom as nes_rom

# Kaggle hiện dùng Python 3.12 + NumPy mới; nes-py giữ header byte dạng np.uint8.
# Khi nes-py tính np.uint8 * 1024, NumPy có thể raise OverflowError.
# Ép sang Python int trước khi nhân để đọc ROM ổn định.
nes_rom.ROM.prg_rom_size = property(lambda self: 16 * int(self.header[4]))
nes_rom.ROM.chr_rom_size = property(lambda self: 8 * int(self.header[5]))

# Contra NES RAM map
_RAM_LEVEL = 0x0030
_RAM_P1_LIVES = 0x0032
_RAM_P1_GAME_OVER = 0x0038
_RAM_BOSS_DEFEATED = 0x003B
_RAM_END_LEVEL_SEQ = 0x002D

_RAM_SCORE_HI = 0x07E2
_RAM_SCORE_LO = 0x07E3

_RAM_SCREEN_NUMBER = 0x0064
_RAM_SCROLL_X = 0x00FD

_RAM_P1_STATE = 0x0090
_RAM_P1_SPRITE_X = 0x0334

_ENEMY_TYPE_BASE = 0x0528
_ENEMY_SLOT_COUNT = 16

_LOGO_WAIT = 300
_START_HOLD = 60
_START_RELEASE = 90

_FRONTIER_SCORE_MARGIN = 32
_STAGNATION_GRACE_STEPS = 90


def _bcd2(byte: int) -> int:
    return (byte >> 4) * 10 + (byte & 0x0F)


class ContraEnv(NESEnv):
    reward_range = (-float('inf'), float('inf'))

    def __init__(self):
        super().__init__(ROM_PATH)
        self._prev_score = 0
        self._prev_lives = 0
        self._prev_x = 0
        self._prev_level = 0
        self._max_x = 0
        self._stagnant_steps = 0

    def _did_reset(self):
        for _ in range(_LOGO_WAIT):
            self._frame_advance(0)
        for _ in range(_START_HOLD):
            self._frame_advance(8)
        for _ in range(_START_RELEASE):
            self._frame_advance(0)

        self._prev_score = self._read_score()
        self._prev_lives = int(self.ram[_RAM_P1_LIVES])
        self._prev_x = self._read_x()
        self._prev_level = int(self.ram[_RAM_LEVEL])
        self._max_x = self._prev_x
        self._stagnant_steps = 0

    def _get_reward(self):
        score_now = self._read_score()
        lives_now = int(self.ram[_RAM_P1_LIVES])
        x_now = self._read_x()
        level_now = int(self.ram[_RAM_LEVEL])
        player_state = int(self.ram[_RAM_P1_STATE])
        just_died = lives_now < self._prev_lives
        x_delta = x_now - self._prev_x
        new_progress = max(0, x_now - self._max_x)

        if not just_died and player_state == 0x01:
            progress_reward = float(
                np.clip(x_delta - 0.5, -4, 3)
                + 1.5 * np.clip(new_progress, 0, 4)
            )
        else:
            progress_reward = 0.0

        near_frontier = x_now >= self._max_x - _FRONTIER_SCORE_MARGIN
        if near_frontier or new_progress > 0:
            score_reward = float(np.clip(score_now - self._prev_score, 0, 1))
        else:
            score_reward = 0.0

        life_penalty = -15.0 if just_died else 0.0

        if not just_died and player_state == 0x01:
            active_enemies = sum(
                1 for i in range(_ENEMY_SLOT_COUNT)
                if self.ram[_ENEMY_TYPE_BASE + i] != 0
            )
            dodge_reward = 0.1 if active_enemies > 0 else 0.0
        else:
            dodge_reward = 0.0

        if player_state == 0x01 and self._stagnant_steps > _STAGNATION_GRACE_STEPS:
            stagnation_penalty = -min(
                2.0,
                (self._stagnant_steps - _STAGNATION_GRACE_STEPS) / 60,
            )
        else:
            stagnation_penalty = 0.0

        if level_now > self._prev_level:
            terminal_reward = 50.0
        elif bool(self.ram[_RAM_P1_GAME_OVER]):
            terminal_reward = -35.0
        else:
            terminal_reward = 0.0

        return (
            progress_reward
            + score_reward
            + life_penalty
            + dodge_reward
            + stagnation_penalty
            + terminal_reward
        ) / 10

    def _get_done(self):
        return bool(self.ram[_RAM_P1_GAME_OVER])

    def _get_info(self):
        return {
            'score': self._read_score(),
            'lives': int(self.ram[_RAM_P1_LIVES]),
            'level': int(self.ram[_RAM_LEVEL]) + 1,
            'x_pos': self._read_x(),
            'camera_x': self._read_camera_x(),
            'player_screen_x': int(self.ram[_RAM_P1_SPRITE_X]),
            'max_x': self._max_x,
            'stagnant_steps': self._stagnant_steps,
            'player_state': int(self.ram[_RAM_P1_STATE]),
            'boss_defeated': bool(self.ram[_RAM_BOSS_DEFEATED]),
            'stage_over': self._stage_is_over(),
            'game_over': bool(self.ram[_RAM_P1_GAME_OVER]),
        }

    def _did_step(self, done):  # noqa: ARG002
        x_now = self._read_x()
        level_now = int(self.ram[_RAM_LEVEL])
        player_state = int(self.ram[_RAM_P1_STATE])

        if player_state == 0x01:
            if level_now > self._prev_level or x_now > self._max_x + 1:
                self._max_x = x_now
                self._stagnant_steps = 0
            elif x_now <= self._prev_x + 0.5:
                self._stagnant_steps += 1
            else:
                self._stagnant_steps = max(0, self._stagnant_steps - 1)
        else:
            self._stagnant_steps = 0

        self._prev_score = self._read_score()
        self._prev_lives = int(self.ram[_RAM_P1_LIVES])
        self._prev_x = x_now
        self._prev_level = level_now

    def _read_score(self) -> int:
        return (_bcd2(self.ram[_RAM_SCORE_HI]) * 100
                + _bcd2(self.ram[_RAM_SCORE_LO])) * 100

    def _read_x(self) -> int:
        return self._read_camera_x() + int(self.ram[_RAM_P1_SPRITE_X])

    def _read_camera_x(self) -> int:
        return int(self.ram[_RAM_SCREEN_NUMBER]) * 256 + int(self.ram[_RAM_SCROLL_X])

    def _stage_is_over(self) -> bool:
        return bool(self.ram[_RAM_END_LEVEL_SEQ])


In [ ]:
# Cell 5 - Gymnasium wrapper giống Contra/wrappers.py

import cv2
import gymnasium as gym
from nes_py.wrappers import JoypadSpace

OBS_SHAPE = (84, 84, 1)
NOOP_ACTION = 0
_P1_STATE_NORMAL = 0x01


class ContraGymnasiumEnv(gym.Env):
    metadata = {"render_modes": ["rgb_array"]}

    def __init__(self, frame_skip: int = 4, render_mode: str | None = None, respawn_noop_frames: int = 90):
        super().__init__()
        self._env = JoypadSpace(ContraEnv(), COMPLEX_MOVEMENT)
        self.frame_skip = frame_skip
        self.render_mode = render_mode
        self.respawn_noop_frames = respawn_noop_frames
        self._respawn_noop_remaining = 0
        self._prev_lives = 0
        self._prev_player_state = _P1_STATE_NORMAL
        self._waiting_for_respawn_landing = False

        self.observation_space = gym.spaces.Box(low=0, high=255, shape=OBS_SHAPE, dtype=np.uint8)
        self.action_space = gym.spaces.Discrete(len(COMPLEX_MOVEMENT))

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        obs = self._env.reset()
        self._respawn_noop_remaining = 0
        self._prev_lives = int(self._ram[_RAM_P1_LIVES])
        self._prev_player_state = int(self._ram[_RAM_P1_STATE])
        self._waiting_for_respawn_landing = False
        return self._preprocess(obs), {}

    def step(self, action):
        total_reward = 0.0
        terminated = False
        obs = None
        info = {}
        forced_noop = False
        executed_action = action

        for _ in range(self.frame_skip):
            action_to_send = NOOP_ACTION if self._should_force_noop() else action
            forced_noop = forced_noop or (action_to_send == NOOP_ACTION and action != NOOP_ACTION)
            executed_action = action_to_send

            obs, reward, done, info = self._env.step(action_to_send)
            total_reward += reward
            self._update_respawn_guard()
            if done:
                terminated = True
                break

        info = dict(info)
        info["forced_respawn_noop"] = forced_noop
        info["respawn_noop_remaining"] = self._respawn_noop_remaining
        info["executed_action"] = int(executed_action)
        return self._preprocess(obs), total_reward, terminated, False, info

    def render(self):
        if self.render_mode == "rgb_array":
            return self._env.render(mode="rgb_array")
        return None

    def close(self):
        self._env.close()

    def _preprocess(self, obs: np.ndarray) -> np.ndarray:
        gray = cv2.cvtColor(obs, cv2.COLOR_RGB2GRAY)
        resized = cv2.resize(gray, (84, 84), interpolation=cv2.INTER_AREA)
        return resized[:, :, np.newaxis]

    @property
    def _ram(self):
        return self._env.unwrapped.ram

    def _should_force_noop(self) -> bool:
        player_state = int(self._ram[_RAM_P1_STATE])
        return player_state != _P1_STATE_NORMAL or self._respawn_noop_remaining > 0

    def _update_respawn_guard(self) -> None:
        lives_now = int(self._ram[_RAM_P1_LIVES])
        player_state = int(self._ram[_RAM_P1_STATE])

        if lives_now < self._prev_lives:
            self._waiting_for_respawn_landing = True
            self._respawn_noop_remaining = 0
        elif (
            self._waiting_for_respawn_landing
            and self._prev_player_state != _P1_STATE_NORMAL
            and player_state == _P1_STATE_NORMAL
        ):
            self._waiting_for_respawn_landing = False
            self._respawn_noop_remaining = self.respawn_noop_frames
        elif self._respawn_noop_remaining > 0 and player_state == _P1_STATE_NORMAL:
            self._respawn_noop_remaining -= 1

        self._prev_lives = lives_now
        self._prev_player_state = player_state


In [ ]:
# Cell 6 - Smoke test env trước khi train

from gymnasium.wrappers import TimeLimit

smoke_env = TimeLimit(ContraGymnasiumEnv(frame_skip=FRAME_SKIP), max_episode_steps=MAX_EPISODE_STEPS)
obs, info = smoke_env.reset(seed=SEED)
print("obs shape:", obs.shape, "dtype:", obs.dtype, "action_space:", smoke_env.action_space)
for i in range(SMOKE_TEST_STEPS):
    obs, reward, terminated, truncated, info = smoke_env.step(smoke_env.action_space.sample())
    print(i, "reward=", round(float(reward), 4), "done=", terminated or truncated, "x=", info.get("x_pos"))
    if terminated or truncated:
        break
smoke_env.close()
print("Smoke test OK")


In [ ]:
# Cell 7 - Build vector env, PPO model, callbacks giống train.py

import os
import numpy as np
from gymnasium.wrappers import TimeLimit
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback, CheckpointCallback
from stable_baselines3.common.utils import set_random_seed
from stable_baselines3.common.vec_env import (
    DummyVecEnv,
    SubprocVecEnv,
    VecFrameStack,
    VecMonitor,
    VecTransposeImage,
)
from tqdm.auto import tqdm


def close_existing_progress_bar():
    old_pbar = globals().get("_KAGGLE_TRAIN_PBAR")
    if old_pbar is not None:
        try:
            old_pbar.close()
        except Exception:
            pass
    globals()["_KAGGLE_TRAIN_PBAR"] = None


def make_env(rank: int, seed: int = 0):
    def _init():
        import warnings
        warnings.filterwarnings("ignore")
        env = ContraGymnasiumEnv(frame_skip=FRAME_SKIP)
        env = TimeLimit(env, max_episode_steps=MAX_EPISODE_STEPS)
        env.reset(seed=seed + rank)
        return env
    set_random_seed(seed)
    return _init


def build_vec_env(n_envs: int, seed: int = 0):
    if n_envs == 1:
        vec_env = DummyVecEnv([make_env(0, seed)])
    else:
        # Kaggle notebook/Linux: fork tránh lỗi pickling khi class được định nghĩa trong cell.
        vec_env = SubprocVecEnv([make_env(i, seed) for i in range(n_envs)], start_method="fork")
    vec_env = VecFrameStack(vec_env, n_stack=4)
    vec_env = VecMonitor(vec_env, LOG_DIR)
    vec_env = VecTransposeImage(vec_env)
    return vec_env


def build_model(vec_env):
    return PPO(
        policy="CnnPolicy",
        env=vec_env,
        learning_rate=1e-4,
        n_steps=512,
        batch_size=128,
        n_epochs=10,
        gamma=0.9,
        gae_lambda=1.0,
        clip_range=0.2,
        ent_coef=0.02,
        vf_coef=0.5,
        max_grad_norm=0.5,
        tensorboard_log=LOG_DIR,
        verbose=1,
    )


class SaveBestCallback(BaseCallback):
    def __init__(self, save_path: str, check_freq: int, verbose: int = 1):
        super().__init__(verbose)
        self.save_path = save_path
        self.check_freq = check_freq
        self.best_mean_reward = -np.inf

    def _on_step(self) -> bool:
        if self.n_calls % self.check_freq != 0:
            return True
        if len(self.model.ep_info_buffer) == 0:
            return True

        mean_reward = np.mean([ep["r"] for ep in self.model.ep_info_buffer])
        if mean_reward > self.best_mean_reward:
            self.best_mean_reward = mean_reward
            self.model.save(os.path.join(self.save_path, "contra_ppo_best"))
            if self.verbose:
                print(f"[best] New best mean reward: {mean_reward:.2f} -> saved")
        return True


class TqdmProgressCallback(BaseCallback):
    def __init__(self):
        super().__init__(verbose=0)
        self.pbar = None
        self.last_num_timesteps = 0

    def _on_training_start(self) -> None:
        close_existing_progress_bar()
        total = int(self.locals.get("total_timesteps", 0))
        remaining = max(total - int(self.model.num_timesteps), 0)
        self.last_num_timesteps = int(self.model.num_timesteps)
        self.pbar = tqdm(total=remaining, desc="Training", unit="steps")
        globals()["_KAGGLE_TRAIN_PBAR"] = self.pbar

    def _on_step(self) -> bool:
        current = int(self.model.num_timesteps)
        delta = current - self.last_num_timesteps
        if self.pbar is not None and delta > 0:
            self.pbar.update(delta)
        self.last_num_timesteps = current
        return True

    def _on_training_end(self) -> None:
        if self.pbar is not None:
            self.pbar.close()
        globals()["_KAGGLE_TRAIN_PBAR"] = None


def build_callbacks(n_envs: int):
    checkpoint_cb = CheckpointCallback(
        save_freq=max(100_000 // n_envs, 1),
        save_path=CHECKPOINT_DIR,
        name_prefix="contra_ppo",
        verbose=1,
    )
    best_cb = SaveBestCallback(
        save_path=BEST_MODEL_DIR,
        check_freq=max(50_000 // n_envs, 1),
        verbose=1,
    )
    callbacks = [checkpoint_cb, best_cb]
    if USE_PROGRESS_BAR:
        callbacks.append(TqdmProgressCallback())
    return callbacks


In [ ]:
# Cell 8 - Train PPO; đây là cell cuối cùng cần chạy để train

print(f"[train] envs={N_ENVS} total_steps={TOTAL_TIMESTEPS:,} seed={SEED}")
close_existing_progress_bar()
train_env = build_vec_env(N_ENVS, SEED)

try:
    if RESUME_PATH:
        print(f"[train] Resuming from {RESUME_PATH}")
        model = PPO.load(RESUME_PATH, env=train_env, tensorboard_log=LOG_DIR)
    else:
        model = build_model(train_env)

    callbacks = build_callbacks(N_ENVS)
    model.learn(
        total_timesteps=TOTAL_TIMESTEPS,
        callback=callbacks,
        tb_log_name="PPO_contra",
        reset_num_timesteps=RESUME_PATH is None,
        progress_bar=False,
    )

    final_path = os.path.join(CHECKPOINT_DIR, "contra_ppo_final")
    model.save(final_path)
    print(f"[train] Done - model saved to {final_path}.zip")
finally:
    train_env.close()
